# Lab 3D — Serverless Model Customization (optional)

**Persona:** Data Scientist · **Lifecycle stage:** Model build · **Run after:** Lab 3C

Labs 3B and 3C fine-tuned Llama 3.2 3B the explicit way: you chose an instance
type, handled GPU capacity, pointed a training channel at a directory, then
evaluated and registered the result as three separate steps.

This lab does the same job with **serverless model customization**, where
SageMaker owns the compute. There is no instance type to pick, no capacity
fallback to write, and training, registration, and MLflow tracking collapse
into one call.

## What you will do

1. Resolve the prompt template from the **MLflow Prompt Registry** — the same
   entry Lab 2A registered and Lab 3B trained against
2. Register Lab 2A's serverless-format dataset as a **versioned `DataSet`**
3. Fine-tune `meta-textgeneration-llama-3-2-3b-instruct` with **LoRA SFT**
4. Confirm the model **auto-registered** into its own model package group,
   still pending approval
5. Read the training curve back out of MLflow

## This lab is optional and self-contained

It depends only on Lab 2A's output. It reads nothing from Lab 3B or 3C and
writes into **its own model package group**, so Lab 4 and Lab 5 are unaffected
whether you run it or skip it.

## What it does *not* do

It registers a model; it does not deploy one. No endpoint is created, so there
is no hourly inference cost to clean up afterwards.


---
## Section 1: Setup

Same environment contract as every other lab — values come from the repo-root
`.env` that `lab0-setup/setup.ipynb` wrote.


In [ ]:
import json
import os
import re
import sys
import time
from datetime import datetime
from pathlib import Path

import boto3
import mlflow
from mlflow.tracking import MlflowClient

from sagemaker.ai_registry.dataset import DataSet
from sagemaker.core.helper.session_helper import Session, get_execution_role
from sagemaker.core.resources import ModelPackageGroup
from sagemaker.core.s3 import S3Uploader
from sagemaker.train.common import TrainingType
from sagemaker.train.sft_trainer import SFTTrainer

# Serverless customization needs sagemaker >= 3.22.0 for sagemaker.train.
from importlib.metadata import version as _pkg_version

_sdk = _pkg_version("sagemaker")
print(f"sagemaker SDK: {_sdk}")
assert tuple(int(p) for p in _sdk.split(".")[:2]) >= (3, 22), (
    f"serverless model customization needs sagemaker>=3.22.0, found {_sdk}. "
    "Re-run lab0-setup/setup.ipynb."
)

sess = Session()
region = sess.boto_region_name
bucket = sess.default_bucket()

# Locate the repo-root .env loader.
for _candidate in [os.getcwd(), *[str(p) for p in Path(os.getcwd()).parents]]:
    if os.path.exists(os.path.join(_candidate, "workshop_env.py")):
        sys.path.insert(0, _candidate)
        break


def env_from_file(key, default=""):
    """Return one key's value from the repo-root .env, or `default`.

    Reads the FILE, not os.environ, on purpose: Studio pre-seeds
    MLFLOW_TRACKING_URI with the domain's DEFAULT MLflow app, and the shared
    loader uses override=False, so an ambient value would silently win over the
    project's own. Labs 2A and 3B resolve it the same way.
    """
    try:
        from workshop_env import find_env

        env_path = find_env()
        if env_path:
            for line in env_path.read_text().splitlines():
                line = line.strip()
                if line.startswith(f"{key}="):
                    return line.split("=", 1)[1].strip().strip('"').strip("'")
    except Exception as exc:
        print(f"  (could not read .env: {type(exc).__name__}: {exc})")
    return default


### The execution role has to be passed explicitly

This is the first place the serverless path differs from Lab 3B in a way that
will stop you. The SDK validates the role's **trust policy** before it submits
anything, and a role that SageMaker cannot assume is rejected up front:

```
RoleValidationError: IAM role '...' cannot be used for 'training' workloads.
Its trust policy does not allow 'sagemaker.amazonaws.com' to assume it
```

In Studio the ambient role is already a SageMaker execution role, so this is
mostly invisible. On a **local kernel** — where your credentials are an SSO or
IAM user role that SageMaker cannot assume — it fails immediately. Reading
`SAGEMAKER_EXEC_ROLE` from `.env` works in both places, which is why every
`DataSet` and trainer call below passes `role=` rather than relying on a
default.


In [ ]:
role = env_from_file("SAGEMAKER_EXEC_ROLE")
role_source = ".env (SAGEMAKER_EXEC_ROLE)"
if not role:
    try:
        role = get_execution_role()
        role_source = "get_execution_role()"
    except Exception as exc:
        raise RuntimeError(
            "Could not resolve a SageMaker execution role. Run "
            "lab0-setup/setup.ipynb to populate SAGEMAKER_EXEC_ROLE in .env. "
            f"get_execution_role() said: {type(exc).__name__}: {exc}"
        ) from exc

# Lab 2A writes under the user-profile prefix; resolve the same one so this lab
# reads the dataset that participant actually produced.
profile_name = env_from_file("SAGEMAKER_USER_PROFILE")
if not profile_name:
    raise RuntimeError(
        "SAGEMAKER_USER_PROFILE is not set in .env. Run lab0-setup/setup.ipynb "
        "-- Lab 2A uses it to build the S3 prefix this lab reads."
    )

profile_name = profile_name[0].upper() + profile_name[1:]

# Lab 2A writes the flat prompt/completion pair this lab needs to a prefix of
# its own, deliberately apart from Lab 3B's "dolly_dataset" training channel:
# JumpStart globs every .jsonl in a channel into one dataset, so a file with
# this lab's schema in there fails Lab 3B's training job on a schema cast.
#
# The name matters, not just the separation. S3 keys have no directories, so
# "dolly_dataset_sft" would still sit INSIDE a channel listed as
# "dolly_dataset" -- string prefix, not folder. "dolly_sft" does not.
DATASET_PREFIX = "dolly_sft"
data_location = f"s3://{bucket}/{profile_name}/{DATASET_PREFIX}"

# Section 10's optional evaluation needs Lab 2A's OTHER output: the held-out
# split and template.json in instruction format. Those stay in the Lab 3B
# channel prefix, so track the two locations separately -- training reads the
# flat pair above, scoring reads the instruction pair here.
INSTRUCT_PREFIX = "dolly_dataset"
instruct_location = f"s3://{bucket}/{profile_name}/{INSTRUCT_PREFIX}"

print(f"Region        : {region}")
print(f"Bucket        : {bucket}")
print(f"Role          : {role}")
print(f"  source      : {role_source}")
print(f"Profile       : {profile_name}")
print(f"Lab 2A output : {data_location}  (SFT pair, for training)")
print(f"              : {instruct_location}  (held-out split, for Section 10)")


### Region check

Serverless model customization is not available everywhere. There is no API to
query availability, so this is a static list that has to be maintained by hand.


In [ ]:
SERVERLESS_REGIONS = {"us-east-1", "us-west-2", "ap-northeast-1", "eu-west-1"}

if region not in SERVERLESS_REGIONS:
    raise RuntimeError(
        f"Serverless model customization is not available in {region}. "
        f"Supported: {', '.join(sorted(SERVERLESS_REGIONS))}. "
        "Lab 3B (instance-based fine-tuning) works in far more regions -- "
        "use that instead, or redeploy the workshop in a supported region."
    )
print(f"\u2713 {region} supports serverless model customization")


---
## Section 2: Why the base model changes

Lab 3B fine-tunes `meta-textgeneration-llama-3-2-3b` — the **base** model.
Serverless customization has no recipe for it. Asking for one fails:

```
ValueError: No recipes found for model 'meta-textgeneration-llama-3-2-3b'
            with customization technique: SFT
```

The same call against the **instruct** variant succeeds, for all four
techniques:

| Model ID | SFT | DPO | RLVR | RLAIF |
|---|---|---|---|---|
| `meta-textgeneration-llama-3-2-3b` | — | — | — | — |
| `meta-textgeneration-llama-3-2-3b-instruct` | yes | yes | yes | yes |

**This changes what the lab demonstrates, not just a string.** Lab 3B starts
from a model that does not reliably follow instructions, so fine-tuning teaches
it the task *and* the response format. Here the starting point already follows
instructions, so SFT is adapting an instruction-follower to your summarization
style and length conventions. Expect a lower starting loss and a smaller
relative improvement — that is the model being better to begin with, not the
training working less well.

The cell below asks the recipe catalogue directly rather than trusting the
table, because recipe availability changes over time.


In [ ]:
from sagemaker.train import list_hyperparameters

BASE_MODEL = "meta-textgeneration-llama-3-2-3b-instruct"
TECHNIQUE = "SFT"
TRAINING_TYPE = TrainingType.LORA

try:
    _hp_catalogue = list_hyperparameters(BASE_MODEL, TECHNIQUE, "LORA")
    print(f"\u2713 {BASE_MODEL}")
    print(f"  {TECHNIQUE} + LORA recipe available")
except ValueError as exc:
    raise RuntimeError(
        f"No {TECHNIQUE} recipe for {BASE_MODEL} in {region}. The serverless "
        "model catalogue changes over time -- check the SageMaker console for "
        f"a currently supported instruct model. Original error: {exc}"
    ) from exc

# Lab 3B's base model, for contrast. Expected to have no recipe.
try:
    list_hyperparameters("meta-textgeneration-llama-3-2-3b", TECHNIQUE, "LORA")
    print("\n  note: the base model now HAS an SFT recipe -- the markdown above is stale")
except ValueError as exc:
    print(f"\n  Lab 3B's base model, for contrast:")
    print(f"    meta-textgeneration-llama-3-2-3b -> {str(exc)[:90]}")


---
## Section 3: Prompt lineage

Lab 2A registered the prompt template in the MLflow Prompt Registry and Lab 3B
trained against it. This lab trains on the *same* template — but it arrives by
a different route, and the difference is worth understanding.

| | Lab 3B (JumpStart) | Lab 3D (serverless) |
|---|---|---|
| Who renders the template | the training container, at train time | Lab 2A, at data-prep time |
| How it reaches training | `template.json` in the training channel | baked into the dataset's `prompt` field |
| Needs `template.json` | yes — the container reads it off disk | no |

Because Lab 2A renders both file pairs from the one registry entry, the prompt
text is byte-identical across the two paths. This cell resolves the registry
entry so the version can be recorded as provenance on the training run — the
same lineage Lab 3C reads back.


In [ ]:
USE_CASE = "text-summarization"
PROMPT_NAME = f"{USE_CASE}-instruct"
PROMPT_ALIAS = "production"

mlflow_uri = env_from_file("MLFLOW_TRACKING_URI")
prompt_uri = ""
prompt_version_number = None
prompt_template_text = ""

if mlflow_uri:
    mlflow.set_tracking_uri(mlflow_uri)
    # Same accessor Labs 2A and 3C use. Note it must be mlflow.genai, not
    # MlflowClient: only the module-level load_prompt accepts link_to_model,
    # and MlflowClient.load_prompt raises TypeError on it.
    import mlflow.genai as genai

    try:
        _prompt = genai.load_prompt(
            f"prompts:/{PROMPT_NAME}@{PROMPT_ALIAS}",
            allow_missing=True,   # returns None instead of raising
            link_to_model=False,  # no active run here; this is only a lookup
        )
    except Exception as exc:
        _prompt = None
        print(f"\u26a0 Prompt lookup failed: {type(exc).__name__}: {str(exc)[:140]}")

    if _prompt is not None:
        prompt_version_number = _prompt.version
        prompt_uri = f"prompts:/{PROMPT_NAME}/{_prompt.version}"
        prompt_template_text = _prompt.template
        print(f"\u2713 Resolved {PROMPT_NAME}@{PROMPT_ALIAS}")
        print(f"  version: {prompt_version_number}")
        print(f"  uri:     {prompt_uri}")
    else:
        print(f"\u26a0 No prompt registered as {PROMPT_NAME}@{PROMPT_ALIAS}.")
        print("  Training still works -- the dataset already carries the rendered")
        print("  prompt text. What is lost is the lineage record: the model package")
        print("  will be stamped UNVERIFIED instead of a prompt version.")
        print("  Re-run Lab 2A's prompt registration cell to restore it.")
else:
    print("\u26a0 No MLFLOW_TRACKING_URI in .env -- skipping prompt lineage.")
    print("  Run lab0-setup/setup.ipynb to populate it.")


---
## Section 4: Register the dataset

Serverless customization does not take an S3 path. It takes a **registered
`DataSet`**, which is a versioned artifact — and the version number is part of
the lineage, in the same way the prompt version is.

Two things to know about what this creates:

- The `DataSet` is stored as **SageMaker Hub content**, so the ARN looks like
  `.../hub-content/<hub-id>/DataSet/<name>/1.0.0`. The first call creates a
  private hub in your account; later calls reuse it.
- The required schema is flat `{"prompt", "completion"}` JSONL. Lab 2A's
  `train_sft.jsonl` / `test_sft.jsonl` are exactly that. The three files Lab 3B
  uses will **not** validate here, which is why Lab 2A emits both pairs.

Registration is fast — measured at **7.8 seconds** for 840 records.


In [ ]:
SFT_TRAIN_FILE = "train_sft.jsonl"
SFT_EVAL_FILE = "test_sft.jsonl"

sft_train_s3 = f"{data_location}/{SFT_TRAIN_FILE}"
sft_eval_s3 = f"{data_location}/{SFT_EVAL_FILE}"

# Confirm Lab 2A produced the serverless-format files before going further --
# a missing file here means Lab 2A ran before this lab existed.
from sagemaker.core.s3 import S3Downloader

_present = set(S3Downloader.list(f"{data_location}/"))
_missing = [p for p in (sft_train_s3, sft_eval_s3) if p not in _present]
if _missing:
    raise FileNotFoundError(
        f"Lab 2A has not produced {', '.join(Path(m).name for m in _missing)} at "
        f"{data_location}. Re-run lab2-data-prep/lab-2a-fine-tuning-data-prep.ipynb "
        "-- the cell titled 'Also render the serverless format' writes them, and "
        "Section 7 uploads them to this prefix. If you ran an earlier version of "
        "Lab 2A they are in the dolly_dataset prefix instead; re-running Lab 2A "
        "moves them here and clears them out of Lab 3B's training channel."
    )

print("Lab 2A objects found:")
for obj in sorted(_present):
    _tag = "  <- used by this lab" if obj in (sft_train_s3, sft_eval_s3) else ""
    print(f"  {obj}{_tag}")


In [ ]:
DATASET_NAME = f"{USE_CASE}-serverless"

# DataSet.create() bumps the version when the name already exists, so re-running
# this lab produces 1.0.0, 1.0.1, ... rather than failing or silently
# overwriting. Each training run therefore pins an exact dataset version.
_t0 = time.time()
train_dataset = DataSet.create(
    name=DATASET_NAME,
    source=sft_train_s3,
    role=role,
    sagemaker_session=sess,
    wait=True,
)
print(f"\n\u2713 Training dataset registered in {time.time() - _t0:.1f}s")
print(f"  arn: {train_dataset.arn}")

_t0 = time.time()
eval_dataset = DataSet.create(
    name=f"{DATASET_NAME}-eval",
    source=sft_eval_s3,
    role=role,
    sagemaker_session=sess,
    wait=True,
)
print(f"\u2713 Validation dataset registered in {time.time() - _t0:.1f}s")
print(f"  arn: {eval_dataset.arn}")


### Why pass a validation dataset at all

`SFTTrainer` has a `train_val_split_ratio` hyperparameter, defaulting to `0.9`,
so it will happily carve its own validation split out of the training data.

This lab passes Lab 2A's `test.jsonl` as an explicit `validation_dataset`
instead. Lab 2A's split is **stratified and leakage-checked** — it asserts no
record appears on both sides. Letting the trainer re-split would throw that
work away and quietly reintroduce the possibility of leakage. When you have a
vetted split, use it.


---
## Section 5: Model package group

The trained model registers itself, so the group has to exist first. Lab 3D
uses **its own group**, separate from the timestamped group Lab 3C creates.

That separation is deliberate. If a serverless LoRA model and a JumpStart model
landed in the same group, "deploy the approved version" would become ambiguous,
and Lab 4 would stop being deterministic.


In [ ]:
MODEL_PACKAGE_GROUP_NAME = f"{USE_CASE}-serverless"

from botocore.exceptions import ClientError

try:
    model_package_group = ModelPackageGroup.create(
        model_package_group_name=MODEL_PACKAGE_GROUP_NAME,
        model_package_group_description=(
            "Lab 3D - serverless LoRA SFT of Llama 3.2 3B Instruct for summarization"
        ),
    )
    print(f"\u2713 Created model package group: {MODEL_PACKAGE_GROUP_NAME}")
except ClientError as exc:
    if exc.response["Error"]["Code"] in ("ResourceInUse", "ValidationException"):
        model_package_group = ModelPackageGroup.get(
            model_package_group_name=MODEL_PACKAGE_GROUP_NAME
        )
        print(f"\u2713 Reusing existing model package group: {MODEL_PACKAGE_GROUP_NAME}")
    else:
        raise

print(f"  arn: {model_package_group.model_package_group_arn}")


---
## Section 6: Configure the trainer

Notice what is **absent** compared with Lab 3B: no `Compute`, no instance type,
no capacity-aware fallback list. SageMaker owns the hardware, so none of those
are decisions you get to make — or have to handle when capacity is short.


In [ ]:
EXPERIMENT_NAME = f"{USE_CASE}-serverless"

trainer_kwargs = dict(
    model=BASE_MODEL,
    training_type=TRAINING_TYPE,
    model_package_group=model_package_group,
    training_dataset=train_dataset.arn,
    validation_dataset=eval_dataset.arn,
    s3_output_path=f"s3://{bucket}/lab3d-serverless/output/",
    sagemaker_session=sess,
    role=role,
)

# MLflow auto-wires even with no arguments -- it creates an experiment named
# after the model package group. Naming it explicitly keeps this lab's runs
# findable next to Lab 3A/3B's, instead of in a separate auto-named experiment.
if mlflow_uri:
    trainer_kwargs["mlflow_resource_arn"] = mlflow_uri
    trainer_kwargs["mlflow_experiment_name"] = EXPERIMENT_NAME
    print(f"MLflow experiment: {EXPERIMENT_NAME}")
else:
    print("MLflow not configured -- training metrics will not be tracked")


### License acceptance

Llama 3.2 3B Instruct is a **gated model**. The SDK enforces this at trainer
construction, not at submission:

```
ValueError: Model 'meta-textgeneration-llama-3-2-3b-instruct' is a gated model
            and requires EULA acceptance. Please set accept_eula=True
```

This model is licensed under the **Llama 3.2 Community License Agreement**.
Review the terms before continuing:
<https://huggingface.co/meta-llama/Llama-3.2-3B-Instruct/blob/main/LICENSE.txt>

**You must read and agree to the license to proceed.** Set `ACCEPT_EULA = True`
in the cell below only if you accept it. Lab 3B asks you the same question for
the base model.


In [ ]:
# Set to True only if you have read and accept the Llama 3.2 Community License
# linked above. Left False, this cell stops the lab here by design.
ACCEPT_EULA = False

if not ACCEPT_EULA:
    raise SystemExit(
        "ACCEPT_EULA is False. Review the Llama 3.2 Community License linked "
        "above and set ACCEPT_EULA = True in this cell to continue."
    )

trainer = SFTTrainer(accept_eula=ACCEPT_EULA, **trainer_kwargs)
print("\u2713 Trainer configured\n")

hp = trainer.hyperparameters
print("Recipe defaults:")
print(f"  global_batch_size     : {hp.global_batch_size}")
print(f"  learning_rate         : {hp.learning_rate}")
print(f"  max_epochs            : {hp.max_epochs}")
print(f"  lora_rank             : {hp.lora_rank}")
print(f"  lora_alpha            : {hp.lora_alpha}")
print(f"  dataset_max_len       : {hp.dataset_max_len}")
print("\nFor every parameter, its range and valid options: hp.get_info()")


### Optional: override hyperparameters

The defaults are what the measured 11.5-minute benchmark below used. Each
override is validated against the recipe, so an out-of-range value fails
immediately with the allowed range rather than wasting a training run.

`max_epochs` defaults to **1**, which is the single biggest lever on both
quality and duration here.


In [ ]:
# Uncomment and adjust. hp.get_info() prints the valid range for each one.

# trainer.hyperparameters.max_epochs = 3            # 1 - 100
# trainer.hyperparameters.learning_rate = 0.0002    # 1e-07 - 0.001
# trainer.hyperparameters.global_batch_size = 128   # 64, 128, 256, 512, 1024
# trainer.hyperparameters.lora_rank = 32            # 8, 16, 32, 64, 128

print(f"max_epochs        = {trainer.hyperparameters.max_epochs}")
print(f"learning_rate     = {trainer.hyperparameters.learning_rate}")
print(f"global_batch_size = {trainer.hyperparameters.global_batch_size}")
print(f"lora_rank         = {trainer.hyperparameters.lora_rank}")


---
## Section 7: Train

Measured on Lab 2A's 840 records with the defaults above:

| Phase | Duration |
|---|---|
| Waiting for capacity | 94s |
| Preparing instances | 31s |
| Downloading input data | 62s |
| Downloading training image | 219s |
| **Training** | **187s** |
| Uploading model | 62s |
| **Total** | **688s (11.5 min)** |

Most of that is not training. Actual gradient work was about three minutes; the
rest is capacity acquisition and a large container image. A bigger dataset
mainly grows the training phase, so the fixed overhead matters less as you
scale up.

The cell submits with `wait=False` and polls, so you get phase-by-phase
progress instead of a blocking call with nothing to look at.


In [ ]:
_t0 = time.time()
training_job = trainer.train(wait=False)
print(f"Submitted in {time.time() - _t0:.1f}s")
print(f"  job name: {training_job.training_job_name}")
print(f"  status  : {training_job.training_job_status}\n")

sm_client = boto3.client("sagemaker", region_name=region)
_seen = set()
_start = time.time()

while True:
    _desc = sm_client.describe_training_job(
        TrainingJobName=training_job.training_job_name
    )
    for _transition in _desc.get("SecondaryStatusTransitions", []):
        _key = (_transition["Status"], _transition.get("StatusMessage", ""))
        if _key not in _seen:
            _seen.add(_key)
            print(f"[{time.time() - _start:6.0f}s] {_transition['Status']:<12} "
                  f"{_transition.get('StatusMessage', '')[:70]}")

    _status = _desc["TrainingJobStatus"]
    if _status != "InProgress":
        print(f"\n{_status} after {time.time() - _start:.0f}s")
        if _desc.get("FailureReason"):
            print(f"FailureReason: {_desc['FailureReason'][:500]}")
        break

    time.sleep(30)

assert _status == "Completed", f"training ended as {_status}, not Completed"
model_artifacts = _desc.get("ModelArtifacts", {}).get("S3ModelArtifacts", "")
print(f"\nModel artifacts: {model_artifacts}")


---
## Section 8: The model registered itself

In Lab 3C you evaluated the model and called `register()` yourself. Here it
already happened — completing the training job created the model package.

Two details matter for governance:

- **`ModelApprovalStatus` is `None`.** Auto-registration does not mean
  auto-approval. The Lab 4 promotion gate still applies, so nothing reaches an
  endpoint without someone approving it.
- **The LoRA adapter is merged into the final model**
  (`merge_lora_on_final_save: True`), so the artifact is a complete model, not
  an adapter you have to compose at load time.


In [ ]:
_packages = sm_client.list_model_packages(
    ModelPackageGroupName=MODEL_PACKAGE_GROUP_NAME,
    SortBy="CreationTime",
    SortOrder="Descending",
    MaxResults=5,
)["ModelPackageSummaryList"]

print(f"Model packages in {MODEL_PACKAGE_GROUP_NAME}:\n")
for _pkg in _packages:
    print(f"  v{_pkg['ModelPackageVersion']:<3} {_pkg['ModelPackageStatus']:<12} "
          f"approval={_pkg.get('ModelApprovalStatus', 'None'):<10} "
          f"{_pkg['CreationTime']:%Y-%m-%d %H:%M}")

latest_model_package_arn = _packages[0]["ModelPackageArn"] if _packages else ""
print(f"\nLatest: {latest_model_package_arn}")
print("\nApproval is still pending -- Lab 4's gate is unchanged by auto-registration.")


### Record the prompt lineage on the model package

The dataset already contains the rendered prompt, so the *model* is tied to the
prompt implicitly. That is not good enough for governance: six months from now
nobody can tell which registry version produced it by looking at the package.

Writing it into `CustomerMetadataProperties` makes the link explicit and
queryable — the same provenance Lab 3C records.

> Metadata values are validated against
> `[\p{L}\p{Z}\p{N}_.:/=+\-@]*`, which rejects characters as ordinary as
> parentheses. `metadata_safe()` strips anything outside that set, matching
> Lab 3C's sanitizer.


In [ ]:
_DISALLOWED = re.compile(r"[^\w\s.:/=+\-@]", re.UNICODE)


def metadata_safe(value):
    """Strip characters CustomerMetadataProperties rejects."""
    return _DISALLOWED.sub("", str(value)).strip()


if latest_model_package_arn:
    _metadata = {
        "use_case": USE_CASE,
        "customization": "serverless-lora-sft",
        "base_model": BASE_MODEL,
        "training_dataset_arn": train_dataset.arn,
        "validation_dataset_arn": eval_dataset.arn,
        "training_job_name": training_job.training_job_name,
        "prompt_name": PROMPT_NAME,
        "prompt_version": str(prompt_version_number) if prompt_version_number else "UNVERSIONED",
        "prompt_version_uri": prompt_uri or "UNVERIFIED-no-registry-entry",
        "produced_by_lab": "lab-3d-serverless-fine-tuning",
    }
    _metadata = {k: metadata_safe(v) for k, v in _metadata.items()}

    sm_client.update_model_package(
        ModelPackageArn=latest_model_package_arn,
        CustomerMetadataProperties=_metadata,
    )
    print("\u2713 Provenance recorded on the model package\n")
    for _k, _v in _metadata.items():
        print(f"  {_k:24} {_v[:70]}")
else:
    print("\u26a0 No model package found -- skipping provenance.")


---
## Section 9: Read the training curve from MLflow

MLflow tracking is automatic. Even with no MLflow arguments at all, the trainer
creates an experiment named after the model package group and logs to it.

Logged metrics go well beyond loss: sequence-length distribution for the
dataset (`data/train_seq_len_*`) and GPU peak memory (`perf/gpu_peak_memory_*`),
which is the useful signal when deciding whether a larger `global_batch_size`
would fit.

The trainer underneath is Verl, so the loss curve is **`train/loss`**. Lab 3B's
JumpStart path logs different names — a reason to list what was actually logged
rather than hardcoding a metric name.


In [ ]:
if not mlflow_uri:
    print("MLflow not configured -- nothing to read.")
else:
    mlflow.set_tracking_uri(mlflow_uri)
    _client = MlflowClient()

    _run = None
    for _exp in _client.search_experiments():
        for _candidate in _client.search_runs([_exp.experiment_id], max_results=100):
            if _candidate.data.tags.get("mlflow.runName") == training_job.training_job_name:
                _run = _candidate
                break
        if _run:
            print(f"Experiment: {_exp.name}")
            break

    if _run is None:
        print(f"No MLflow run named {training_job.training_job_name} found yet.")
        print("Metrics can lag job completion by a few seconds -- re-run this cell.")
    else:
        print(f"Run id    : {_run.info.run_id}\n")
        _loss = [m for m in _run.data.metrics if "loss" in m.lower()]
        print(f"Loss metrics logged: {sorted(_loss)}\n")

        for _metric in ("train/loss", "train/per_token_loss"):
            if _metric in _run.data.metrics:
                _history = _client.get_metric_history(_run.info.run_id, _metric)
                print(f"  {_metric}: {len(_history)} steps, "
                      f"{_history[0].value:.4f} -> {_history[-1].value:.4f}")

        for _metric in sorted(_run.data.metrics):
            if _metric.startswith("perf/"):
                print(f"  {_metric}: {_run.data.metrics[_metric]:.2f}")


In [ ]:
# Plot whichever loss curve was logged, rather than assuming a metric name.
if mlflow_uri and _run is not None:
    import matplotlib.pyplot as plt

    _available = set(_run.data.metrics)
    _candidates = [("train/loss", "total_loss"), ("val/loss", "val_eval_total_loss")]
    _to_plot = [next(n for n in names if n in _available)
                for names in _candidates if _available & set(names)]

    if not _to_plot:
        print(f"No expected loss curves. Logged: {sorted(_available)}")
    else:
        _fig, _axes = plt.subplots(1, len(_to_plot),
                                   figsize=(6 * len(_to_plot), 3.5), squeeze=False)
        for _ax, _metric in zip(_axes[0], _to_plot):
            _history = _client.get_metric_history(_run.info.run_id, _metric)
            _ax.plot([h.step for h in _history], [h.value for h in _history],
                     linewidth=2, marker="o", markersize=4)
            _ax.set_xlabel("Step")
            _ax.set_ylabel("Loss")
            _ax.set_title(_metric, fontweight="bold")
            _ax.grid(True, alpha=0.3)
        plt.suptitle(f"Lab 3D - {training_job.training_job_name}", fontweight="bold")
        plt.tight_layout()
        plt.show()


---
## Section 10: Evaluate the model (optional)

Section 8 registered a model with **no quality metrics attached**. A reviewer
looking at that package has the provenance but no evidence, so there is nothing
to approve on. This section supplies the evidence.

It is opt-in because it costs money: scoring a summarization model means
generating summaries, which means a GPU endpoint. Leave `RUN_EVALUATION = False`
and Lab 3D stays a cheap mechanics lab that creates nothing billable.

### Three constraints worth understanding before the code

**Metrics cannot be attached to the package.** `ModelMetrics` is settable only
at `create_model_package` time, and here the *training job* created the package.
`update_model_package` rejects the field outright:

```
ParamValidationError: Unknown parameter in input: "ModelMetrics", must be one of:
ModelPackageArn, ModelApprovalStatus, ModelPackageRegistrationType,
ApprovalDescription, CustomerMetadataProperties, ...
```

So scores go to MLflow and to `CustomerMetadataProperties`. This is a real
difference from Lab 3C, where you build the package yourself and can pass
`ModelMetrics` at creation.

**We deploy the artifact, not the package.** `CreateModel` from a model package
requires `Approved` status:

```
ValidationException: Invalid approval status "null" ... Can only create a
SageMaker Model for ModelPackages in Approved status.
```

Approving in order to evaluate would invert the governance gate — approval is
supposed to be the *consequence* of evidence. So this section serves the
training artifact directly and leaves the package pending until Section 11.

**The artifact has a servable sub-directory.** The training output holds FSDP
rank shards and optimiser state, which are not servable, alongside
`checkpoints/hf_merged/` — a complete HF Llama model. That sub-directory is
what gets deployed.


In [ ]:
# ---------------------------------------------------------------------------
# Gates. Both default to off; nothing below runs until you opt in.
# ---------------------------------------------------------------------------
RUN_EVALUATION = False       # deploys ONE GPU endpoint, deleted at the end
COMPARE_AGAINST_BASE = False # deploys a SECOND endpoint for a true delta

EVAL_SAMPLES = 20            # same count Lab 3C scores, so numbers compare

# Resolve the current LMI container from the SDK rather than pinning a tag.
#
# This matters more than it looks. Lab 3B's container
# (djl-inference:0.29.0-lmi11.0.0-cu124) CANNOT serve this artifact: the
# serverless recipe writes its merged checkpoint with the transformers v5
# config schema -- `rope_parameters` and `dtype` where 4.x expects
# `rope_scaling` and `torch_dtype` (the config reports
# transformers_version 5.3.0). The older container reads it, guesses, and
# crash-loops with:
#
#     ValueError: Unknown RoPE scaling type extended
#
# Resolving the image keeps this lab aligned with whatever the recipe emits.
from sagemaker.core import image_uris

SERVING_IMAGE = image_uris.retrieve(framework="djl-lmi", region=region)

# SAGEMAKER_PROGRAM / SAGEMAKER_SUBMIT_DIRECTORY are deliberately omitted:
# hf_merged/ ships no inference.py and no code/ directory, so pointing the
# container at one would fail. LMI's default handler loads HF_MODEL_ID.
SERVING_ENV = {
    "HF_MODEL_ID": "/opt/ml/model",
    "OPTION_MAX_MODEL_LEN": "8192",
    "ENDPOINT_SERVER_TIMEOUT": "3600",
    "SAGEMAKER_MODEL_SERVER_TIMEOUT": "3600",
    "SAGEMAKER_MODEL_SERVER_WORKERS": "1",
}

# A 3B model in bf16 fits a single 24 GB GPU. Capacity for any one type is
# genuinely unreliable, so walk a list -- the same approach Lab 3B uses.
INSTANCE_FALLBACK = ["ml.g6.xlarge", "ml.g5.xlarge",
                     "ml.g6.2xlarge", "ml.g5.2xlarge"]

if not RUN_EVALUATION:
    print("RUN_EVALUATION is False -- skipping evaluation.")
    print("  The model stays registered with provenance but no metrics, so a")
    print("  reviewer has nothing to approve on. Set RUN_EVALUATION = True to")
    print("  deploy a temporary endpoint and produce the evidence.")
else:
    merged_uri = model_artifacts.rstrip("/") + "/checkpoints/hf_merged/"
    print(f"Will deploy: {merged_uri}")
    print(f"  image    : {SERVING_IMAGE}")
    print(f"  instances: {INSTANCE_FALLBACK}")
    print(f"  samples  : {EVAL_SAMPLES}")
    print(f"  baseline : {'instruct base endpoint' if COMPARE_AGAINST_BASE else 'Lab 3C recorded metrics'}")


### Deploy, score, tear down

The helper walks the instance list on `InsufficientInstanceCapacity` — which is
not hypothetical, it happened repeatedly while this lab was being written — and
every endpoint is deleted in a `finally`, so an exception part-way through does
not leave a GPU running.


In [ ]:
import contextlib


def _capacity_error(text):
    t = (text or "").lower()
    return ("insufficientinstancecapacity" in t
            or "provision requested ml compute capacity" in t
            or "resourcelimitexceeded" in t)


def _wait_terminal(name, timeout=2700):
    t0 = time.time()
    while time.time() - t0 < timeout:
        try:
            d = sm_client.describe_endpoint(EndpointName=name)
        except sm_client.exceptions.ClientError:
            return "Gone", ""
        if d["EndpointStatus"] not in ("Creating", "Updating", "SystemUpdating"):
            return d["EndpointStatus"], d.get("FailureReason", "") or ""
        time.sleep(30)
    return "Timeout", ""


def _drop(endpoint=None, config=None, model=None):
    """Best-effort teardown. Waits out Creating -- DeleteEndpoint is rejected
    while an endpoint is still coming up, which is how a stray GPU gets left
    running."""
    if endpoint:
        st, _ = _wait_terminal(endpoint, timeout=1800)
        if st != "Gone":
            with contextlib.suppress(Exception):
                sm_client.delete_endpoint(EndpointName=endpoint)
                print(f"  deleted endpoint {endpoint}")
    if config:
        with contextlib.suppress(Exception):
            sm_client.delete_endpoint_config(EndpointConfigName=config)
    if model:
        with contextlib.suppress(Exception):
            sm_client.delete_model(ModelName=model)


def deploy_for_eval(label, s3_uri, accept_eula=False):
    """Serve an HF model directory. Returns (endpoint_name, instance_type)."""
    stamp = datetime.now().strftime("%Y%m%d%H%M%S")
    model_name = f"lab3d-eval-{label}-{stamp}"
    source = {"S3Uri": s3_uri, "S3DataType": "S3Prefix", "CompressionType": "None"}
    if accept_eula:
        # Gated JumpStart weights refuse CreateModel without this.
        source["ModelAccessConfig"] = {"AcceptEula": True}

    sm_client.create_model(
        ModelName=model_name, ExecutionRoleArn=role,
        PrimaryContainer={"Image": SERVING_IMAGE, "Environment": SERVING_ENV,
                          "ModelDataSource": {"S3DataSource": source}})
    _eval_teardown["models"].append(model_name)

    for instance in INSTANCE_FALLBACK:
        endpoint = f"{model_name}-{instance.replace('.', '-')}"
        config = f"{endpoint}-cfg"
        print(f"  trying {label} on {instance}")
        sm_client.create_endpoint_config(
            EndpointConfigName=config,
            ProductionVariants=[{
                "VariantName": "AllTraffic", "ModelName": model_name,
                "InstanceType": instance, "InitialInstanceCount": 1,
                "ModelDataDownloadTimeoutInSeconds": 2400,
                "ContainerStartupHealthCheckTimeoutInSeconds": 2400}])
        _eval_teardown["configs"].append(config)
        sm_client.create_endpoint(EndpointName=endpoint, EndpointConfigName=config)
        status, reason = _wait_terminal(endpoint)
        if status == "InService":
            print(f"  {label} InService on {instance}")
            _eval_teardown["endpoints"].append(endpoint)
            return endpoint, instance
        print(f"  {label} on {instance}: {status} -- {reason[:140]}")
        _drop(endpoint=endpoint, config=config)
        if not _capacity_error(reason):
            raise RuntimeError(f"{label} failed for a non-capacity reason: {reason[:300]}")
    raise RuntimeError(f"no capacity for {label} on any of {INSTANCE_FALLBACK}")


def generate(endpoint, prompt_text, eula=False):
    payload = {"inputs": prompt_text, "parameters": {"max_new_tokens": 100}}
    kwargs = dict(EndpointName=endpoint, ContentType="application/json",
                  Accept="application/json", Body=json.dumps(payload))
    if eula:
        kwargs["CustomAttributes"] = "accept_eula=true"
    body = boto3.client("sagemaker-runtime", region_name=region).invoke_endpoint(**kwargs)
    out = json.loads(body["Body"].read().decode())
    if isinstance(out, list) and len(out) == 1:
        out = out[0]
    return out.get("generated_text", f"<unexpected shape: {json.dumps(out)[:120]}>")


_eval_teardown = {"endpoints": [], "configs": [], "models": []}
print("helpers defined" if RUN_EVALUATION else "helpers defined (evaluation gated off)")


In [ ]:
# ---------------------------------------------------------------------------
# Generate predictions. Prompts are built from the SAME registered template the
# model trained on -- a mismatch here silently depresses every metric.
# ---------------------------------------------------------------------------
eval_rows = []

if RUN_EVALUATION:
    # Check the scoring inputs BEFORE deploying anything. Finding a missing
    # object after deploy_for_eval() means waiting out a GPU endpoint you are
    # already paying for, only to fail on a download.
    _needed = [f"{instruct_location}/test.jsonl"]
    if not prompt_template_text:
        _needed.append(f"{instruct_location}/template.json")
    _have = set(S3Downloader.list(f"{instruct_location}/"))
    _absent = [u for u in _needed if u not in _have]
    if _absent:
        raise FileNotFoundError(
            "Section 10 scores against Lab 2A's instruction-format held-out "
            f"split, which is missing: {_absent}. Re-run "
            "lab2-data-prep/lab-2a-fine-tuning-data-prep.ipynb, or set "
            "RUN_EVALUATION = False to skip scoring."
        )
    try:
        ft_endpoint, ft_instance = deploy_for_eval(
            "tuned", model_artifacts.rstrip("/") + "/checkpoints/hf_merged/")

        base_endpoint = None
        if COMPARE_AGAINST_BASE:
            # The instruct base this model was fine-tuned FROM. Not the base
            # model Lab 3B uses -- that is a different checkpoint, and scoring
            # against it would conflate the fine-tune with a model swap.
            _hub = sm_client.describe_hub_content(
                HubName="SageMakerPublicHub", HubContentType="Model",
                HubContentName=BASE_MODEL, HubContentVersion="2.17.0")
            _doc = _hub["HubContentDocument"]
            _doc = json.loads(_doc) if isinstance(_doc, str) else _doc
            base_endpoint, _ = deploy_for_eval("base", _doc["HostingArtifactUri"],
                                               accept_eula=True)

        # Lab 2A's held-out split, in instruction format: scoring compares
        # against `response`, so it comes from the instruct prefix rather than
        # the flat pair the trainer consumed.
        _local_test = Path("test_eval.jsonl")
        S3Downloader.download(f"{instruct_location}/test.jsonl", ".")
        Path("test.jsonl").rename(_local_test)
        _records = [json.loads(l) for l in
                    _local_test.read_text().splitlines() if l.strip()][:EVAL_SAMPLES]

        # Render with the registry template if it resolved, else the S3 copy.
        if prompt_template_text:
            _tpl_prompt = re.sub(r"\{\{\s*(\w+)\s*\}\}", r"{\1}", prompt_template_text)
        else:
            S3Downloader.download(f"{instruct_location}/template.json", ".")
            _tpl_prompt = json.loads(Path("template.json").read_text())["prompt"]

        print(f"\nscoring {len(_records)} samples")
        for _i, _rec in enumerate(_records):
            _prompt = _tpl_prompt.format(**_rec) + "\n\n### Response:\n"
            _row = {"prompt": _prompt, "target": _rec["response"],
                    "tuned": generate(ft_endpoint, _prompt)}
            if base_endpoint:
                _row["base"] = generate(base_endpoint, _prompt, eula=True)
            eval_rows.append(_row)
            if (_i + 1) % 5 == 0:
                print(f"  {_i + 1}/{len(_records)}")

        print(f"\u2713 collected {len(eval_rows)} predictions")
        print(f"  mean length  tuned {sum(len(r['tuned']) for r in eval_rows)/len(eval_rows):.0f}"
              + (f"  base {sum(len(r['base']) for r in eval_rows)/len(eval_rows):.0f}"
                 if base_endpoint else "")
              + f"  target {sum(len(r['target']) for r in eval_rows)/len(eval_rows):.0f} chars")

        if base_endpoint:
            # Same guard Lab 3C now carries: if every generation matches, the
            # comparison is vacuous and the metrics will agree to the decimal.
            _same = sum(1 for r in eval_rows if r["base"] == r["tuned"])
            print(f"  identical generations: {_same}/{len(eval_rows)}")
            assert _same < len(eval_rows), (
                "base and fine-tuned returned identical text for every prompt; "
                "the comparison below would be meaningless")
    finally:
        # Endpoints go away even if generation raised part-way through.
        for _e in _eval_teardown["endpoints"]:
            _drop(endpoint=_e)
        for _c in _eval_teardown["configs"]:
            _drop(config=_c)
        for _m in _eval_teardown["models"]:
            _drop(model=_m)
        if RUN_EVALUATION:
            print("\u2713 evaluation endpoints torn down")
else:
    print("skipped (RUN_EVALUATION is False)")


### Score with the same metric code Lab 3C uses

This imports `metrics.py` — the shared helpers — and calls `mlflow.evaluate`
exactly as Lab 3C does. That is deliberate: scoring 3D with a different
implementation would make its numbers incomparable to 3C's, and comparing
against 3C is the cheap alternative to standing up a second endpoint.


In [ ]:
eval_metrics = {}

if eval_rows:
    import pandas as pd
    from metrics import bleu, rouge1, rouge2, rougeL   # shared with Lab 3C

    mlflow.set_tracking_uri(mlflow_uri)
    mlflow.set_experiment(EXPERIMENT_NAME)

    _df_tuned = pd.DataFrame({"predictions": [r["tuned"] for r in eval_rows],
                              "targets": [r["target"] for r in eval_rows]})
    _stamp = datetime.now().strftime("%Y%m%d-%H%M%S")

    with mlflow.start_run(run_name=f"serverless-tuned-eval-{_stamp}"):
        mlflow.log_param("base_model", BASE_MODEL)
        mlflow.log_param("customization", "serverless-lora-sft")
        mlflow.log_param("training_job", training_job.training_job_name)
        if prompt_uri:
            mlflow.log_param("prompt_version_uri", prompt_uri)
        _res_tuned = mlflow.evaluate(data=_df_tuned, targets="targets",
                                     predictions="predictions",
                                     extra_metrics=[bleu, rouge1, rouge2, rougeL])
        # Keep the predictions, not just the aggregates -- Lab 3C logged none
        # and a suspect run could not be diagnosed afterwards.
        mlflow.log_table(data=pd.DataFrame(eval_rows),
                         artifact_file="predictions.json")
    eval_metrics["tuned"] = {k: _res_tuned.metrics[k]
                             for k in ("bleu", "rouge1", "rouge2", "rougeL")}

    if "base" in eval_rows[0]:
        _df_base = pd.DataFrame({"predictions": [r["base"] for r in eval_rows],
                                 "targets": [r["target"] for r in eval_rows]})
        with mlflow.start_run(run_name=f"serverless-base-eval-{_stamp}"):
            mlflow.log_param("base_model", BASE_MODEL)
            mlflow.log_param("customization", "none (base instruct model)")
            _res_base = mlflow.evaluate(data=_df_base, targets="targets",
                                        predictions="predictions",
                                        extra_metrics=[bleu, rouge1, rouge2, rougeL])
        eval_metrics["base"] = {k: _res_base.metrics[k]
                                for k in ("bleu", "rouge1", "rouge2", "rougeL")}

    print("Scores:")
    for _which, _m in eval_metrics.items():
        print(f"  {_which:6} " + "  ".join(f"{k}={v:.4f}" for k, v in _m.items()))
else:
    print("no predictions to score")


### Verdict

Computed from the numbers, never asserted. With `COMPARE_AGAINST_BASE = True`
the baseline is the instruct model this one was fine-tuned from. Otherwise the
baseline is whatever Lab 3C recorded for the JumpStart fine-tune — a cross-lab
comparison that answers a more interesting question: **does serverless LoRA SFT
on the instruct model beat JumpStart instruction-tuning on the base model?**

The confound in that version is real and worth naming: the two runs start from
different checkpoints. Same test split and same prompt template, though, so the
comparison is fair on the data even if the starting points differ.


In [ ]:
if eval_metrics:
    _baseline, _label = None, ""
    if "base" in eval_metrics:
        _baseline, _label = eval_metrics["base"], "instruct base (this lab)"
    elif mlflow_uri:
        # Pull Lab 3C's recorded fine-tuned scores for a cross-lab baseline.
        _client = MlflowClient()
        _best = None
        for _exp in _client.search_experiments():
            for _run in _client.search_runs([_exp.experiment_id], max_results=100):
                if not _run.data.tags.get("mlflow.runName", "").startswith("fine-tuned-model-eval"):
                    continue
                if {"bleu", "rouge1", "rouge2", "rougeL"} <= set(_run.data.metrics):
                    if _best is None or _run.info.start_time > _best.info.start_time:
                        _best = _run
        if _best is not None:
            _baseline = {k: _best.data.metrics[k]
                         for k in ("bleu", "rouge1", "rouge2", "rougeL")}
            _label = f"Lab 3C fine-tune ({_best.data.tags['mlflow.runName']})"

    if _baseline is None:
        print("No baseline available -- absolute scores only:")
        for _k, _v in eval_metrics["tuned"].items():
            print(f"  {_k:8} {_v:.4f}")
        print("\nRun Lab 3C, or set COMPARE_AGAINST_BASE = True, for a comparison.")
    else:
        print(f"baseline: {_label}\n")
        print(f"{'metric':<10}{'baseline':>10}{'this model':>13}{'delta':>10}")
        print("-" * 43)
        _wins = []
        for _k in ("bleu", "rouge1", "rouge2", "rougeL"):
            _b, _t = _baseline[_k], eval_metrics["tuned"][_k]
            _wins.append(_t > _b)
            _flag = "better" if _t > _b else ("same" if _t == _b else "WORSE")
            print(f"{_k:<10}{_b:>10.4f}{_t:>13.4f}{_t - _b:>+10.4f}  {_flag}")
        _n = sum(_wins)
        print()
        if _n == 4:
            print("VERDICT: better on all 4 metrics -- evidence supports approval.")
        elif _n == 0:
            print("VERDICT: not better on any metric. Do not approve on this evidence.")
        else:
            print(f"VERDICT: mixed -- better on {_n} of 4. Decide which metric matters")
            print("  for this use case and record that reasoning on the model card.")
else:
    print("nothing to compare")


---
## Section 11: Record the evidence and decide

`ModelMetrics` is create-only, so the scores go into
`CustomerMetadataProperties` alongside the provenance from Section 8. Not as
tidy as the native field, but it is queryable and it travels with the package.

Approval stays **manual and explicit**. `APPROVE_MODEL` defaults to `False`
because a notebook that approves its own model is not a governance gate.


In [ ]:
APPROVE_MODEL = False   # set True only after reading the verdict above

if latest_model_package_arn and eval_metrics:
    _evidence = {
        "eval_samples": str(len(eval_rows)),
        "eval_dataset": f"{instruct_location}/test.jsonl",
        "eval_metric_code": "metrics.py (shared with lab-3c)",
    }
    for _k, _v in eval_metrics["tuned"].items():
        _evidence[f"eval_{_k}"] = f"{_v:.4f}"
    if "base" in eval_metrics:
        for _k, _v in eval_metrics["base"].items():
            _evidence[f"baseline_{_k}"] = f"{_v:.4f}"

    sm_client.update_model_package(
        ModelPackageArn=latest_model_package_arn,
        CustomerMetadataProperties={k: metadata_safe(v) for k, v in _evidence.items()})
    print("\u2713 evaluation evidence recorded on the model package")
    for _k, _v in sorted(_evidence.items()):
        print(f"  {_k:24} {_v}")
elif latest_model_package_arn:
    print("No metrics to record (evaluation was skipped).")

if APPROVE_MODEL and latest_model_package_arn:
    sm_client.update_model_package(
        ModelPackageArn=latest_model_package_arn,
        ModelApprovalStatus="Approved",
        ApprovalDescription=(
            f"Approved on eval evidence: "
            + ", ".join(f"{k}={v:.4f}" for k, v in eval_metrics.get("tuned", {}).items())
        )[:1024])
    print("\n\u2713 model package APPROVED -- it can now be deployed")
    print("  Note: CreateModel from a package only works once it is Approved,")
    print("  which is why evaluation above served the artifact directly.")
else:
    _status = "unknown"
    if latest_model_package_arn:
        _status = sm_client.describe_model_package(
            ModelPackageName=latest_model_package_arn).get("ModelApprovalStatus", "None")
    print(f"\nApproval status: {_status}")
    print("  Set APPROVE_MODEL = True to approve, or approve in the SageMaker")
    print("  console / via Lab 4's pipeline.")


---
## Section 12: What this lab traded away

Side by side with Labs 3B and 3C:

| | Lab 3B + 3C | Lab 3D |
|---|---|---|
| Compute | you pick the instance type | SageMaker owns it |
| GPU capacity | you write the fallback logic | not your problem |
| Techniques | whatever the container supports | SFT, DPO, RLVR, RLAIF recipes only |
| Base models | the whole JumpStart catalogue | models with a serverless recipe |
| Prompt delivery | `template.json` in the channel | rendered into the dataset |
| Dataset | S3 path | versioned `DataSet` artifact |
| Registration | explicit, in Lab 3C | automatic |
| Metrics on the package | `ModelMetrics` at creation | metadata only (create-only field) |
| Deploying to evaluate | any registered version | package must be Approved first |
| MLflow | you wire it up | automatic |
| Custom container | supported | not supported |

**The trade is control for operational surface.** Lab 3D removes real work —
instance sizing, capacity handling, registration plumbing — and in exchange you
can only train what a recipe already covers. For a standard LoRA SFT on a
supported model, that is a good deal. For a custom loss function, a bespoke
container, or an unsupported base model, Lab 3B is the only path.

**On cost:** `describe-training-job` returns `BillableTimeInSeconds: 0` and
`ResourceConfig: None` for serverless jobs. There are no instance-seconds to
report, so per-run cost cannot be read from the API — check Cost Explorer after
charges settle, or the serverless customization pricing page.

### What is left behind

No endpoint is created, so nothing is billing hourly. These do persist:

- the registered model package (pending approval)
- the `DataSet` versions, held as private SageMaker Hub content
- model artifacts under `s3://{bucket}/lab3d-serverless/output/`

All cheap to keep. The teardown cell below is commented out on purpose —
deleting a registered model is a governance action, not cleanup.


In [ ]:
# Teardown. Commented out deliberately: deleting a registered model package is
# a governance decision, so make it explicitly rather than by running a cell.

# for _pkg in sm_client.list_model_packages(
#         ModelPackageGroupName=MODEL_PACKAGE_GROUP_NAME)["ModelPackageSummaryList"]:
#     sm_client.delete_model_package(ModelPackageName=_pkg["ModelPackageArn"])
# sm_client.delete_model_package_group(ModelPackageGroupName=MODEL_PACKAGE_GROUP_NAME)
# DataSet.delete_by_name(name=DATASET_NAME, sagemaker_session=sess)
# DataSet.delete_by_name(name=f"{DATASET_NAME}-eval", sagemaker_session=sess)

print("Lab 3D complete.")
print(f"  model package group : {MODEL_PACKAGE_GROUP_NAME}")
print(f"  training job        : {training_job.training_job_name}")
print(f"  dataset             : {train_dataset.arn.split('/')[-2]} v{train_dataset.arn.split('/')[-1]}")
print(f"  prompt              : {prompt_uri or 'not resolved'}")
